# Notebook 06 — Power BI Data Export

## Reading this notebook

NB06 is the terminal notebook. It re-exports values computed in NB03 and NB04 as machine-readable artifacts for Power BI consumption. **It does not compute anything new**. Every numeric value in the exports comes from the NB03/NB04 contracts.

The critical distinction from **NB03/NB04/NB05: NB06's output is consumed by people who did not see the notebooks**. A dashboard user sees 'fact_fairness_testing.csv' and 'fact_scenario_analysis.csv' without the surrounding context. The data dictionary is the artifact that carries that context forward — it is the only thing in the export folder that can prevent the numbers from being mis-interpreted.

**What this notebook is for:**

The previous notebooks produce static artifacts — printed reports, saved
charts, JSON summaries. Those are the right format for regulatory submission
and audit. But a Chief Risk Officer reviewing the portfolio, or a business
manager exploring segment performance, needs something more interactive.

Power BI provides that interactivity. This notebook's job is to export the
data in a shape Power BI can consume: fact tables, dimension tables, and a
summary metrics file.

**What it produces:**

Nine CSV files in `powerbi_exports/`:

- `fact_loan_portfolio.csv` — loan-level data for drill-down analysis
- `fact_segment_analysis.csv` — aggregated results by FICO, LTV, property type,
  and loan age
- `fact_scenario_analysis.csv` — results for the four stress scenarios
- `fact_capital_adequacy.csv` — capital impact by scenario
- `fact_ifrs9_stages.csv` — IFRS 9 stage distribution
- `fact_fairness_testing.csv` — fairness testing results
- `dim_date.csv` — date dimension for time-series visualizations
- `dim_segment.csv` — segment type dimension
- `summary_metrics.csv` — key performance indicators for a dashboard header

**The staging + promotion mechanism fixes a real v1.1 bug**. In v1.1, 'fact_capital_adequacy.csv' and 'dim_date.csv' survived from previous runs because the notebook wrote directly to EXPORT_DIR without cleaning. The v1.2 mechanism writes to '_staging' first, deletes any file in EXPORT_DIR not in 'PROMOTED_SET', copies the staged files forward, recomputes the manifest against EXPORT_DIR, and asserts the final state. The narrative should state this as a positive: "Stale artifacts from previous runs are cleaned up automatically, and the promotion assertion prevents a partial export from being left in place."

**The fairness export's** 'Interpretation' **column is the most important text in the export folder**. Three values: 'UNRELIABLE — high cardinality (>50)', 'DI≥0.80', 'DI<0.80'. A dashboard user who sees 'DI_Ratio = 0.00' and does not read the 'Interpretation' column will misread the result. The 'Pass_4_5ths' boolean (True/False) is present for filtering, but a filter on 'Pass_4_5ths = False' that does not also filter on 'Interpretation = "UNRELIABLE..."' will show misleading results. The data dictionary addresses this (the "NOTE — Fairness testing" section); the narrative should reinforce it.

**The capital coverage export is the reframed metric**. The column is 'ECL_Pct_of_Assets', not 'ECL_to_CET1'. The ECL/CET1 ratio is not in the export — the export carries only the meaningful coverage number. A dashboard user who wants the ECL/CET1 diagnostic must read the NB04 contract directly, not the export. The narrative should say: "The export carries the meaningful coverage metric. The ECL/CET1 scale diagnostic is deliberately not exported, because it is not a coverage metric and a dashboard user would misread it."

**The data dictionary's LIMITATIONS section is the context that travels with the numbers**. Eight limitations listed (100K sample, US data as Canadian proxy, constructed LGD, proxy SICR, fairness unreliable at high cardinality, etc.). A dashboard user who reads only the CSVs and not the data dictionary is missing the context that makes the numbers interpretable.

In [1]:
# Notebook 06 — Power BI Data Export
# Version 1.2  [EDIT E-N06-VER] bumped from 1.1
# Changes vs 1.1:
#   E-N06-VER    version bump
#   E-N06-01     results read from NB03/NB04 contracts (reconciliation assert)
#   E-N06-02     staging directory + promotion of exactly 8 spec'd exports (fixes stale fact_capital_adequacy.csv / dim_date.csv)
#   E-N06-03     canonical load + FileNotFoundError guard
#   E-N06-04     fairness status neutral — no "PASS"/"FAIL" in summary_metrics
#   E-N06-05     data dictionary covers every emitted file, plus dimensional and manifest artifacts (previously undocumented)
#   E-N06-06     C5 scope guard — no ROC/AUC import or panel
#   E-N06-07     run_id / notebook_version on every export; manifest included
#   E-N06-08     dim_segment emitted as a real artifact (was only in spec)
#
# >> NOTE: This is the final notebook of the Fannie Mae pipeline. Its job is to export the analysis results as machine-readable artifacts for Power BI
# >> consumption.
# >>
# >> The critical distinction from all previous notebooks: NB06 does NOT produce new analysis. It RE-EXPORTS values computed in NB03 and NB04.
# >> That's why every numeric value in the exports comes from the contract JSONs (E-N06-01) rather than from re-deriving against the dataframe.
# >>
# >> The OTHER critical distinction: NB06's output is consumed by people who did NOT see the notebooks. A Power BI dashboard user sees
# >> fact_fairness_testing.csv and fact_scenario_analysis.csv without the surrounding context. The data dictionary (E-N06-05) is what carries
# >> that context forward — it's the only artifact in the export folder that can travel with the numbers and prevent their mis-interpretation.
# >>
# >> What breaks if you change this notebook without updating downstream:
# >> - Nothing downstream reads NB06's outputs. This is the terminal notebook.
# >> - BUT: whoever maintains the Power BI dashboard reads them. Changing column names, adding/removing files, or changing the data dictionary
# >>   format breaks the dashboard's expectations.
# >> - The staging + promotion mechanism (E-N06-02) means any stale files from previous runs are cleaned up. If you add a new export, add it to
# >>   SPEC_EXPORTS or the promotion assertion will fail.
# >>
# >> What this notebook does NOT do:
# >> - Compute anything. All values come from NB03/NB04 contracts.
# >> - Interpret the exports. The data dictionary documents what's in each file; the analyst interprets it in the Validation Opinion.
# >> - Declare fairness results as "PASS" or "FAIL". The interpretation column says "UNRELIABLE — high cardinality" or "DI≥0.80" / "DI<0.80". No verdict tokens.
# >> - Export capital adequacy. The coverage diagnostic is exported, but labelled as coverage in the data dictionary.
# >> - Include ROC/AUC panels. C5 scope guard (E-N06-06).

In [2]:
import pandas as pd
import numpy as np
import os
import sys
import shutil
import glob
import json
import hashlib
import logging
from datetime import datetime
from pathlib import Path

# >> NOTE: Same version-bump discipline as NB01–NB05. NB06's version appears in every export file, the manifest, and the data dictionary header. Any
# >> change to the export set, column schema, or data dictionary is a MINOR bump; a change to the promotion mechanism or the C5 guard is a MAJOR
# >> bump (because those are the safety mechanisms).
NOTEBOOK_VERSION = "1.2"

# >> NOTE: [E-N06-03] Three canonical paths. The clean data (for row count and the fact_loan_portfolio export), the NB03 contract, and the NB04
# >> contract. All three must exist for the notebook to run.
CLEAN_PATH = 'data/fannie_mae_final_clean.csv'
NB03_PARAMS = 'outputs/notebook_03_parameters.json'
NB04_RESULTS = 'outputs/notebook_04_results.json'

# >> NOTE: [E-N06-02] The staging + promotion mechanism. Every export is written to STAGING_DIR first, then promoted to EXPORT_DIR at the end.
# >> The old files in EXPORT_DIR are deleted if they're not in SPEC_EXPORTS.
# >>
# >> This is the mechanism that fixes the v1.1 bug where stale artifacts (fact_capital_adequacy.csv, dim_date.csv) survived from previous runs.
# >> The v1.1 notebook just wrote files into EXPORT_DIR without cleaning; the "fix" is to stage everything first, then promote atomically.
EXPORT_DIR = 'powerbi_exports'
STAGING_DIR = f'{EXPORT_DIR}/_staging'

# >> NOTE: [E-N06-02] The exact set of files this notebook is allowed to emit. Nine files total (8 exports + data_dictionary.md). If you add a
# >> new export, add it here — the promotion assertion at the end checks that EXPORT_DIR contains exactly this set (plus manifest.csv).
# >>
# >> [E-N06-08] dim_segment.csv is now in the set. In v1.1 it was in the specification but the notebook never produced it. A Power BI dashboard
# >> that tried to load dim_segment would fail. The E-N06-08 fix is to produce it.
SPEC_EXPORTS = {
    'fact_loan_portfolio.csv',
    'fact_segment_analysis.csv',
    'fact_scenario_analysis.csv',
    'fact_capital_coverage.csv',
    'fact_ifrs9_stages.csv',
    'fact_fairness_testing.csv',
    'summary_metrics.csv',
    'dim_segment.csv',
    'data_dictionary.md',
}

### Setup Audit Trail

In [3]:
def compute_file_hash(path):
    """SHA-256-16 of a file, chunked read.

    >> NOTE: Same function as NB01–NB05. Used for the manifest and for individual export hashes. Every export's hash is recorded in the
    >> audit log, so a reviewer can verify no file was modified after export.
    """
    sha256 = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            sha256.update(chunk)
    return sha256.hexdigest()[:16]


def setup_audit_logger(run_id=None):
    """Same pattern as NB01–NB05.

    >> NOTE: Distinct log filename (powerbi_export_audit_trail_{run_id}.log). The log records every file written to staging, every file promoted,
    >> and every stale file removed. A reviewer can reconstruct the full export sequence from the log.
    """
    if run_id is None:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs('logs', exist_ok=True)
    logger = logging.getLogger(f"powerbi_export_{run_id}")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    handler = logging.FileHandler(
        f"logs/powerbi_export_audit_trail_{run_id}.log", encoding='utf-8')
    handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(console_handler)
    return logger, run_id


audit_logger, RUN_ID = setup_audit_logger()
audit_logger.info("=" * 60)
audit_logger.info(f"POWER BI EXPORT v{NOTEBOOK_VERSION}")
audit_logger.info(f"RUN ID: {RUN_ID}")
audit_logger.info("=" * 60)

# >> NOTE: [E-N06-02] The staging directory is created fresh on every run. If a previous run left STAGING_DIR behind (e.g., crashed before
# >> promotion), shutil.rmtree cleans it up. This ensures the staging directory starts empty, so the file count at promotion time is exactly
# >> what this run produced.
# >>
# >> If you ever want to preserve a staging directory from a failed run for debugging, you'd need to comment out the shutil.rmtree here. That's a
# >> MINOR bump — the behavior change is documented. 
if os.path.exists(STAGING_DIR):
    shutil.rmtree(STAGING_DIR)
os.makedirs(STAGING_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

2026-10-22 13:58:51 | INFO | ============================================================
2026-10-22 13:58:51 | INFO | POWER BI EXPORT v1.2
2026-10-22 13:58:51 | INFO | RUN ID: 20261022_135851
2026-10-22 13:58:51 | INFO | ============================================================


### SECTION 1: LOAD DATA & ASSUMPTIONS

In [4]:
# >> NOTE: [E-N06-03] The pre-flight check. Three files must exist:
# >> - CLEAN_PATH: the cleaned dataframe (used for the loan portfolio export)
# >> - NB03_PARAMS: the calibration contract (used for stages, PD/LGD/EAD)
# >> - NB04_RESULTS: the stress+fairness contract (used for scenarios, fairness, capital coverage)
# >>
# >> Same pattern as NB05: fail loudly if any is missing. A partial export is worse than no export, because a dashboard built on partial data looks
# >> complete but isn't.
for p in [CLEAN_PATH, NB03_PARAMS, NB04_RESULTS]:
    if not Path(p).exists():
        raise FileNotFoundError(
            f"Upstream artifact missing: {p}. "
            f"Run Notebooks 01–04 (v1.2+) before Notebook 06."
        )

df = pd.read_csv(CLEAN_PATH)

# >> NOTE: [E-N06-01] Read from NB03/NB04 contracts, not re-derive. Same discipline as E-N05-03. Every numeric value in the exports comes from
# >> these JSONs.
with open(NB03_PARAMS) as f:
    nb03 = json.load(f)
with open(NB04_RESULTS) as f:
    nb04 = json.load(f)

# >> NOTE: The reconciliation assertion. NB04's `upstream_nb03_run_id` must match NB03's `run_id`. If they don't, the exports would mix parameters
# >> from one NB03 run with results from another — a contamination that would silently corrupt every number in the dashboard.
# >>
# >> Same assertion as NB05. It's duplicated here because NB06 is an independent notebook — if it doesn't assert, a developer could run NB06
# >> against mismatched contracts and the export would still succeed.
assert nb04['upstream_nb03_run_id'] == nb03['run_id'], (
    f"Contamination risk: NB04 built on NB03 run {nb04['upstream_nb03_run_id']} "
    f"but NB06 is reading NB03 run {nb03['run_id']}."
)

# >> NOTE: The contract values, extracted for use in the exports. These are the authoritative numbers — same as what NB05 uses in the compliance
# >> report. If a reviewer compares the compliance report to the Power BI export, the numbers must match.
base_pd = nb03['results']['unconditional_pd']
base_lgd = nb03['results']['average_lgd']
base_ead = nb03['results']['average_ead']
stage_distribution = nb03['results']['stage_distribution']
total_weighted_ecl = nb04['total_weighted_ecl']

audit_logger.info(
    f"Loaded NB03 contract run_id={nb03['run_id']}; "
    f"NB04 contract run_id={nb04['run_id']}"
)

print(f"✅ Loaded: {df.shape}, Default Rate: {df['default'].mean():.2%}")
print(f"✅ NB03 contract: PD={base_pd:.2%} LGD={base_lgd:.2%} EAD=${base_ead:,.0f}")

# >> NOTE: The two staged-save helpers. Both write to STAGING_DIR (not EXPORT_DIR). The promotion happens at the end of the notebook (Section 5).
# >>
# >> _write_staged_csv automatically adds run_id and notebook_version columns if they're missing. Every fact table therefore carries provenance.
# >>
# >> _write_staged_text writes a text file (used only for the data dictionary). Text files don't get provenance columns, but the data
# >> dictionary's header includes the run_id and version explicitly.
def _write_staged_csv(dataframe, filename):
    """Write a DataFrame to staging; promotion happens at the end.

    >> NOTE: Do NOT use this helper to write to EXPORT_DIR directly. The staging + promotion mechanism (E-N06-02) is what ensures stale files
    >> are cleaned up. Writing directly to EXPORT_DIR would bypass that.
    >>
    >> The .copy() before adding columns means the caller's dataframe isn't modified. This matters if the same dataframe is written twice with
    >> different filenames (e.g., fact_segment_analysis and fact_scenario_analysis might share a dataframe).
    """
    out = dataframe.copy()
    if 'run_id' not in out.columns:
        out['run_id'] = RUN_ID
    if 'notebook_version' not in out.columns:
        out['notebook_version'] = NOTEBOOK_VERSION
    path = os.path.join(STAGING_DIR, filename)
    out.to_csv(path, index=False)
    audit_logger.info(f"Staged: {filename} hash={compute_file_hash(path)}")


def _write_staged_text(lines, filename):
    """Write a text file to staging.

    >> NOTE: Only used for data_dictionary.md. If you add another text artifact, use this helper. Same staging discipline.
    """
    path = os.path.join(STAGING_DIR, filename)
    with open(path, 'w', encoding='utf-8') as f:
        f.write("\n".join(lines))
    audit_logger.info(f"Staged: {filename} hash={compute_file_hash(path)}")

2026-10-22 13:58:52 | INFO | Loaded NB03 contract run_id=20261022_133307; NB04 contract run_id=20261022_134434
✅ Loaded: (100000, 175), Default Rate: 0.95%
✅ NB03 contract: PD=0.95% LGD=44.28% EAD=$204,745


### SECTION 2: FACT TABLE - LOAN PORTFOLIO

In [5]:
# >> NOTE: This section produces the seven fact tables. Each has a specific purpose for the Power BI dashboard:
# >> - fact_loan_portfolio: one row per loan, for drill-down
# >> - fact_segment_analysis: aggregated by segment, for segment views
# >> - fact_scenario_analysis: one row per scenario, for scenario views
# >> - fact_capital_coverage: coverage diagnostics, for the coverage view
# >> - fact_ifrs9_stages: stage distribution, for the stage pie
# >> - fact_fairness_testing: fairness results, for the fairness panel
# >> - summary_metrics: key metrics, for the dashboard header
# >>
# >> Every fact table is written through _write_staged_csv, which adds run_id and notebook_version. A dashboard user can verify the numbers
# >> against a specific run.

# --- Fact 1: Loan Portfolio ---
# >> NOTE: The loan portfolio export. One row per loan with origination characteristics, derived risk drivers (pd_estimate, lgd_final, ead_final),
# >> and banded versions for the dashboard's segment filters.
# >>
# >> The band definitions (fico_band, ltv_band, age_band) must match NB02's banding for consistency. The bins here are hardcoded to match FDL §2,
# >> but if NB02's bands ever change, this file must update too.
loan_fact = df[['default', 'ORG_UPB', 'ORG_LTV', 'DTI', 'FICO_BOR', 'AGE',
                'PROP_TYPE', 'OCCU_STAT']].copy()
loan_fact['default_flag'] = loan_fact['default'].map({0: 'Performing', 1: 'Default'})
loan_fact['fico_band'] = pd.cut(loan_fact['FICO_BOR'],
                                 bins=[620, 680, 720, 760, 832],
                                 labels=['Low', 'Medium', 'High', 'Very High'])
loan_fact['ltv_band'] = pd.cut(loan_fact['ORG_LTV'],
                                bins=[0, 60, 80, 100],
                                labels=['Low', 'Medium', 'High'])


def age_band(age):
    """Bucket loan age into IFRS 9 age bands.

    >> NOTE: The bins (0-12, 12-24, 24-36, 36-60, 60-120, 120+) are the standard IFRS 9 age bands. They align with NB02's age_group bins and
    >> with the age adjustments in NB03's LGD calibration.
    >>
    >> The 120m+ bucket catches everything older than 10 years. For a 30-year mortgage, this is meaningful — the tail of the age distribution.
    """
    if age < 12: return '0-12m'
    elif age < 24: return '12-24m'
    elif age < 36: return '24-36m'
    elif age < 60: return '36-60m'
    elif age < 120: return '60-120m'
    else: return '120m+'


loan_fact['age_band'] = loan_fact['AGE'].apply(age_band)

# >> NOTE: The derived risk drivers (pd_estimate, lgd_final, ead_final) are carried forward from the clean data. They were computed in NB01 (for the
# >> initial estimates) and NB03 (for the final LGD/EAD). The values in the clean CSV are the ones NB03 used for calibration.
# >>
# >> If you change NB01's pd_estimate construction, the clean CSV changes, and this export reflects the new values. Downstream dashboards might
# >> show different numbers, but the numbers still trace to the pipeline.
if 'pd_estimate' in df.columns:
    loan_fact['pd_estimate'] = df['pd_estimate']
if 'lgd_final' in df.columns:
    loan_fact['lgd_final'] = df['lgd_final']
if 'ead_final' in df.columns:
    loan_fact['ead_final'] = df['ead_final']

_write_staged_csv(loan_fact, 'fact_loan_portfolio.csv')
print(f"✅ Staged: fact_loan_portfolio.csv ({len(loan_fact):,} rows)")


# --- Fact 2: Segment Analysis ---
# >> NOTE: The segment analysis aggregation. For each band category, compute the default rate and count. This is what the dashboard uses for its
# >> segment comparison charts.
# >>
# >> Three segment types: FICO, LTV, Property Type. Each has its own aggregation loop because the categories differ. If you add a new
# >> segmentation (e.g., age band), add a new loop here.
segment_data = []
for band in ['Low', 'Medium', 'High', 'Very High']:
    subset = loan_fact[loan_fact['fico_band'] == band]
    if len(subset) > 0:
        segment_data.append({
            'Segment_Type': 'FICO', 'Category': band,
            'Default_Rate': subset['default'].mean(), 'Count': len(subset),
        })
for band in ['Low', 'Medium', 'High']:
    subset = loan_fact[loan_fact['ltv_band'] == band]
    if len(subset) > 0:
        segment_data.append({
            'Segment_Type': 'LTV', 'Category': band,
            'Default_Rate': subset['default'].mean(), 'Count': len(subset),
        })
for prop in loan_fact['PROP_TYPE'].unique():
    subset = loan_fact[loan_fact['PROP_TYPE'] == prop]
    if len(subset) > 0:
        segment_data.append({
            'Segment_Type': 'Property Type', 'Category': prop,
            'Default_Rate': subset['default'].mean(), 'Count': len(subset),
        })

df_segments = pd.DataFrame(segment_data)
_write_staged_csv(df_segments, 'fact_segment_analysis.csv')
print(f"✅ Staged: fact_segment_analysis.csv ({len(df_segments)} rows)")

# --- Fact 3: Scenario Analysis (from NB04 contract) ---
# >> NOTE: The scenario analysis comes DIRECTLY from NB04's contract. We do not recompute — we read the scenario ECLs from nb04['scenarios'] and
# >> reshape them into a flat table. The numbers are guaranteed to match NB04's analysis.
# >>
# >> If a reviewer wants to verify "did NB06 export the same scenario ECLs that NB04 computed?", the answer is yes by construction — we read the
# >> same JSON.
scenario_rows = []
for name, r in nb04['scenarios'].items():
    scenario_rows.append({
        'Scenario': name,
        'PD': r['pd'],
        'LGD': r['lgd'],
        'ECL': r['ecl'],
        'Weight': r['weight'],
        'Weighted_ECL': r['weighted_ecl'],
    })

df_scenarios = pd.DataFrame(scenario_rows)
_write_staged_csv(df_scenarios, 'fact_scenario_analysis.csv')
print(f"✅ Staged: fact_scenario_analysis.csv ({len(df_scenarios)} rows)")

# --- Fact 4: Capital Coverage (from NB04 contract) ---
# >> NOTE: The capital coverage export. This is the reframed metric (E-N04-03). The column name is ECL_Pct_of_Assets, not ECL_to_CET1, because that's the
# >> meaningful coverage metric. If someone wants the ECL/CET1 diagnostic, it's in the NB04 contract but NOT in this export — the export only
# >> carries the meaningful coverage number.
# >>
# >> The Total_Assets value (1 billion) is an ASSUMPTION from NB04. It's the denominator for the coverage metric. A dashboard user sees
# >> "ECL_Pct_of_Assets" and needs to know the denominator is assumed. The data dictionary documents this.
coverage_rows = []
for name, r in nb04['scenarios'].items():
    portfolio_ecl = r['ecl'] * nb04['capital_coverage']['portfolio_size']
    coverage_rows.append({
        'Scenario': name,
        'Portfolio_ECL': portfolio_ecl,
        'Total_Assets': nb04['capital_coverage']['total_assets'],
        'ECL_Pct_of_Assets': (portfolio_ecl / nb04['capital_coverage']['total_assets']) * 100,
    })

df_coverage = pd.DataFrame(coverage_rows)
_write_staged_csv(df_coverage, 'fact_capital_coverage.csv')
print(f"✅ Staged: fact_capital_coverage.csv (reframed, coverage only)")

# --- Fact 5: IFRS 9 Stages (from NB03 contract) ---
# >> NOTE: The stage distribution export. The Percentage values come from NB03's contract (the actual stage proportions). The Count values are
# >> reconstructed by multiplying by len(df) — this is a derived value, not a contract value, but it matches what NB03 computed (since NB03 used
# >> the same dataframe).
# >>
# >> If you ever want Count to come from NB03's contract directly, add it to the contract JSON. For now, the reconstruction is safe because the
# >> pipeline uses the same df throughout.
stage_rows = []
for stage in ['Stage 1', 'Stage 2', 'Stage 3']:
    pct = stage_distribution.get(stage, 0.0)
    stage_rows.append({
        'Stage': stage,
        'Percentage': pct,
        'Count': int(round(pct * len(df))),
    })
df_stages = pd.DataFrame(stage_rows)
_write_staged_csv(df_stages, 'fact_ifrs9_stages.csv')
print(f"✅ Staged: fact_ifrs9_stages.csv")

# --- Fact 6: Fairness Testing (from NB04 contract; neutral interpretation) ---
# >> NOTE: [E-N06-04] The fairness export. The most important thing about this export is what it does NOT contain: "PASS" and "FAIL" tokens.
# >>
# >> The Interpretation column has three possible values:
# >> - 'UNRELIABLE — high cardinality (>50)': cardinality > 50, DI is degenerate, do not interpret
# >> - 'DI≥0.80': DI in acceptable range
# >> - 'DI<0.80': DI below acceptable threshold
# >>
# >> The Pass_4_5ths column is a BOOLEAN (True/False), not a string. A boolean is a fact, not a verdict — it says "DI is above/below 0.80" without
# >> saying "the model passed/failed". A dashboard can filter on the boolean without the export carrying a claim.
# >>
# >> The Interpretation column carries the crucial context: "UNRELIABLE" for high-cardinality results. A dashboard that filters on Pass_4_5ths=False
# >> without reading the Interpretation column would mislead the user.
fairness_rows = []
for attr, r in nb04['fairness'].items():
    fairness_rows.append({
        'Attribute': attr,
        'Cardinality': r['cardinality'],
        'DI_Ratio': r['di_ratio'],
        'Pass_4_5ths': bool(r['di_ratio'] >= 0.80),
        'Interpretation': r['interpretation'],
    })
df_fairness = pd.DataFrame(fairness_rows)
_write_staged_csv(df_fairness, 'fact_fairness_testing.csv')
print(f"✅ Staged: fact_fairness_testing.csv")

# --- Fact 7: Summary Metrics ---
# >> NOTE: [E-N06-04] The summary metrics export. This is where the "PASS"/"FAIL" token risk was highest in v1.1 — the summary row for fairness
# >> status could have said "PASS" or "FAIL" as a single token.
# >>
# >> The v1.2 fix: the status is a neutral label:
# >> - "TESTED — see data dictionary" if fairness was tested at all
# >> - "NOT ASSESSED — no protected attributes" if it wasn't
# >>
# >> Neither label is a verdict. "TESTED" just says the notebook ran the test. "NOT ASSESSED" says the test couldn't run. Neither says whether
# >> the result is good or bad.
# >>
# >> A dashboard user who sees "Fairness Testing Status: TESTED" knows to look at fact_fairness_testing.csv for the results. A dashboard user
# >> who sees "NOT ASSESSED" knows the analysis was limited.
fairness_tested = len(nb04['fairness']) > 0
fairness_status = (
    "TESTED — see data dictionary" if fairness_tested
    else "NOT ASSESSED — no protected attributes"
)

summary_metrics = pd.DataFrame({
    'Metric': [
        'Total Loans', 'Default Rate',
        'PD', 'LGD', 'EAD',
        'ECL (per loan, weighted)',
        'Stage 1 %', 'Stage 2 %', 'Stage 3 %',
        'Fairness Testing Status',
    ],
    'Value': [
        f"{len(df):,}",
        f"{df['default'].mean():.2%}",
        f"{base_pd:.2%}",
        f"{base_lgd:.2%}",
        f"${base_ead:,.0f}",
        f"${total_weighted_ecl:,.2f}",
        f"{stage_distribution.get('Stage 1', 0):.1%}",
        f"{stage_distribution.get('Stage 2', 0):.1%}",
        f"{stage_distribution.get('Stage 3', 0):.1%}",
        fairness_status,
    ],
})
_write_staged_csv(summary_metrics, 'summary_metrics.csv')
print(f"✅ Staged: summary_metrics.csv")

# --- Dimension table: dim_segment ---
# >> NOTE: [E-N06-08] The dimension table. In v1.1 it was in the specification but never produced — a dashboard that tried to load dim_segment would
# >> fail. The v1.2 fix is to produce it as a real artifact.
# >>
# >> The five rows describe the five segment types used in the exports:
# >> FICO, LTV, Property Type, Loan Age, DTI. A dashboard can join these descriptions to the Segment_Type column in fact_segment_analysis for
# >> human-readable segment names.
dim_segment = pd.DataFrame([
    {'Segment_Type': 'FICO',          'Segment_Description': 'Credit quality bands: Low / Medium / High / Very High'},
    {'Segment_Type': 'LTV',           'Segment_Description': 'Loan-to-value bands: Low / Medium / High'},
    {'Segment_Type': 'Property Type', 'Segment_Description': 'Property categories: SF / PU / CO / MH / CP'},
    {'Segment_Type': 'Loan Age',      'Segment_Description': 'Age bands: 0-12m / 12-24m / 24-36m / 36-60m / 60-120m / 120m+'},
    {'Segment_Type': 'DTI',           'Segment_Description': 'Debt-to-income categories: ≤43% / >43%'},
])
_write_staged_csv(dim_segment, 'dim_segment.csv')
print(f"✅ Staged: dim_segment.csv")

2026-10-22 13:58:53 | INFO | Staged: fact_loan_portfolio.csv hash=d84351cbbd7f88c8
✅ Staged: fact_loan_portfolio.csv (100,000 rows)
2026-10-22 13:58:53 | INFO | Staged: fact_segment_analysis.csv hash=7b594e316ecfdc9d
✅ Staged: fact_segment_analysis.csv (12 rows)
2026-10-22 13:58:53 | INFO | Staged: fact_scenario_analysis.csv hash=28915a89759f0b58
✅ Staged: fact_scenario_analysis.csv (4 rows)
2026-10-22 13:58:53 | INFO | Staged: fact_capital_coverage.csv hash=fb41fdc7c5a307a4
✅ Staged: fact_capital_coverage.csv (reframed, coverage only)
2026-10-22 13:58:53 | INFO | Staged: fact_ifrs9_stages.csv hash=d6d2fdf4e63abbdf
✅ Staged: fact_ifrs9_stages.csv
2026-10-22 13:58:53 | INFO | Staged: fact_fairness_testing.csv hash=8455f21c60abcbdb
✅ Staged: fact_fairness_testing.csv
2026-10-22 13:58:53 | INFO | Staged: summary_metrics.csv hash=5cfb26141c711150
✅ Staged: summary_metrics.csv
2026-10-22 13:58:53 | INFO | Staged: dim_segment.csv hash=e006491fd0a23002
✅ Staged: dim_segment.csv


### SECTION 3: DATA DICTIONARY

In [6]:
# >> NOTE: [E-N06-05] The data dictionary. This is the single most important artifact in the export folder — it's the context that travels with the
# >> numbers to the dashboard user who never sees the notebooks.
# >>
# >> Every column in every export is documented here. Every limitation is documented here. The fairness and capital coverage framing notes are
# >> here — so a dashboard user reading fact_fairness_testing.csv knows what "UNRELIABLE" means and why some DIs are unreliable.
# >>
# >> In v1.1, the data dictionary was incomplete — it documented only some columns and missed the LIMITATIONS section entirely. A dashboard user
# >> in v1.1 could see "DI_Ratio: 0.00%" and conclude the model was unfair, without knowing that high-cardinality DI is degenerate.
# >>
# >> The v1.2 fix: complete documentation. Every file, every column, every limitation, every framing note. If you add a new export, add its
# >> section here. If you add a new column, add its row. If you add a new limitation, add it to the LIMITATIONS section.

data_dict = [
    "=" * 80,
    "POWER BI EXPORTS — DATA DICTIONARY",
    f"Run ID: {RUN_ID}",
    f"Notebook Version: {NOTEBOOK_VERSION}",
    f"NB03 contract run_id: {nb03['run_id']}",
    f"NB04 contract run_id: {nb04['run_id']}",
    "=" * 80,
    "",
    "## fact_loan_portfolio.csv",
    "One row per loan. Origination and derived risk characteristics.",
    "",
    "| Column | Type | Description |",
    "|--------|------|-------------|",
    "| default | int | 1 = default, 0 = performing |",
    "| ORG_UPB | float | Original unpaid principal balance |",
    "| ORG_LTV | float | Original loan-to-value ratio (%) |",
    "| DTI | float | Debt-to-income ratio (%) |",
    "| FICO_BOR | int | Borrower FICO at origination |",
    "| AGE | float | Loan age (months) |",
    "| PROP_TYPE | str | Property type (SF, PU, CO, MH, CP) |",
    "| OCCU_STAT | str | Occupancy status (P, S, I) |",
    "| default_flag | str | 'Performing' or 'Default' |",
    "| fico_band | str | FICO band (Low, Medium, High, Very High) |",
    "| ltv_band | str | LTV band (Low, Medium, High) |",
    "| age_band | str | Age band (0-12m, 12-24m, etc.) |",
    "| pd_estimate | float | Estimated probability of default |",
    "| lgd_final | float | Estimated loss given default |",
    "| ead_final | float | Estimated exposure at default |",
    "| run_id | str | Reproducibility identifier |",
    "| notebook_version | str | Version of the pipeline |",
    "",
    "## fact_segment_analysis.csv",
    "One row per (Segment_Type, Category). Default rates by segment.",
    "",
    "| Column | Type | Description |",
    "|--------|------|-------------|",
    "| Segment_Type | str | FICO / LTV / Property Type |",
    "| Category | str | Segment value |",
    "| Default_Rate | float | Default rate within segment |",
    "| Count | int | Loans in segment |",
    "",
    "## fact_scenario_analysis.csv",
    "One row per scenario (baseline, adverse, severely_adverse, tail_risk).",
    "Values read from NB04 contract — NOT recomputed here.",
    "",
    "| Column | Type | Description |",
    "|--------|------|-------------|",
    "| Scenario | str | Scenario name |",
    "| PD | float | Scenario PD (capped at 0.20) |",
    "| LGD | float | Scenario LGD (capped at 0.90) |",
    "| ECL | float | Scenario ECL per loan |",
    "| Weight | float | Probability weight |",
    "| Weighted_ECL | float | ECL × weight |",
    "",
    "## fact_capital_coverage.csv",
    "Coverage diagnostic — NOT capital adequacy. Shows portfolio ECL as % of total assets.",
    "",
    "| Column | Type | Description |",
    "|--------|------|-------------|",
    "| Scenario | str | Scenario name |",
    "| Portfolio_ECL | float | Scenario ECL × portfolio size |",
    "| Total_Assets | float | Assumed total assets (from NB04 contract) |",
    "| ECL_Pct_of_Assets | float | ECL as % of total assets |",
    "",
    "## fact_ifrs9_stages.csv",
    "One row per IFRS 9 stage. Values from NB03 contract.",
    "",
    "| Column | Type | Description |",
    "|--------|------|-------------|",
    "| Stage | str | Stage 1 / Stage 2 / Stage 3 |",
    "| Percentage | float | Proportion of portfolio |",
    "| Count | int | Loan count |",
    "",
    "## fact_fairness_testing.csv",
    "Fairness testing results. Values from NB04 contract.",
    "",
    "| Column | Type | Description |",
    "|--------|------|-------------|",
    "| Attribute | str | Protected attribute or geographic proxy |",
    "| Cardinality | int | Number of groups |",
    "| DI_Ratio | float | Disparate impact ratio |",
    "| Pass_4_5ths | bool | DI ≥ 0.80 |",
    "| Interpretation | str | 'UNRELIABLE — high cardinality (>50)' if cardinality > 50; otherwise 'DI≥0.80' or 'DI<0.80' |",
    "",
    "## summary_metrics.csv",
    "Key metrics for dashboard display. Fairness Testing Status is a neutral label:",
    "  * 'TESTED — see data dictionary'",
    "  * 'NOT ASSESSED — no protected attributes'",
    "",
    "## dim_segment.csv",
    "Segment dimension table.",
    "",
    "| Column | Type | Description |",
    "|--------|------|-------------|",
    "| Segment_Type | str | FICO / LTV / Property Type / Loan Age / DTI |",
    "| Segment_Description | str | Human-readable band description |",
    "",
    "## manifest.csv",
    "Reproducibility manifest — hash + size for every file in this directory.",
    "",
    "## LIMITATIONS",
    "- All metrics are derived from a 100,000-row random sample of Fannie Mae data.",
    "- US data is used as a proxy for a Canadian bank portfolio.",
    "- LGD is constructed as a function of PD (not empirically calibrated).",
    "- SICR uses a proxy rule (not true SICR).",
    "- Fairness testing on geographic proxies is unreliable at high cardinality.",
    "",
    "## NOTE — Fairness testing",
    "The 4/5ths rule is designed for low-cardinality attributes. High cardinality",
    "(>50 groups) produces degenerate results. The Interpretation column indicates", "whether a result is UNRELIABLE.",
    "",
    "## NOTE — Capital coverage",
    "This is a coverage diagnostic (ECL as % of total assets). It is NOT a capital", "adequacy measure. Capital adequacy requires risk-weighted assets.",
]

_write_staged_text(data_dict, 'data_dictionary.md')
print("✅ Staged: data_dictionary.md")

2026-10-22 13:58:53 | INFO | Staged: data_dictionary.md hash=8190416894e0e6dc
✅ Staged: data_dictionary.md


### SECTION 4: EXPORT MANIFEST

In [7]:
# >> NOTE: [E-N06-07] The manifest for the staged files. Every file in STAGING_DIR gets a hash. The manifest itself is excluded to avoid
# >> recursion.
# >>
# >> This manifest is TEMPORARY — it's discarded when the staging directory is cleaned up at the end. The FINAL manifest is regenerated in Section 5
# >> after promotion (with hashes of the files actually in EXPORT_DIR).
# >>
# >> The temporary manifest serves one purpose: to verify that the staging step produced exactly the expected files before promotion. If a staging
# >> step crashed mid-way, the temporary manifest would list fewer files than SPEC_EXPORTS — catching the problem before it propagates.
manifest_rows = []
for fname in sorted(os.listdir(STAGING_DIR)):
    if fname == 'manifest.csv':
        continue
    fpath = os.path.join(STAGING_DIR, fname)
    if not os.path.isfile(fpath):
        continue
    manifest_rows.append({
        'file': fname,
        'hash_sha256_16': compute_file_hash(fpath),
        'size_bytes': os.path.getsize(fpath),
        'run_id': RUN_ID,
        'notebook_version': NOTEBOOK_VERSION,
        'timestamp': datetime.now().isoformat(),
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(os.path.join(STAGING_DIR, 'manifest.csv'), index=False)
print(f"✅ Staged: manifest.csv ({len(manifest_df)} files hashed)")
audit_logger.info(f"Manifest: {len(manifest_rows)} files hashed")

✅ Staged: manifest.csv (9 files hashed)
2026-10-22 13:58:53 | INFO | Manifest: 9 files hashed


### SECTION 5: PROMOTE STAGING → EXPORT_DIR

In [8]:
# >> NOTE: [E-N06-02] The promotion step. This is where the staging + promotion mechanism does its work:
# >>
# >> 1. PROMOTED_SET is SPEC_EXPORTS plus manifest.csv — the exact set of files that should be in EXPORT_DIR after promotion.
# >>
# >> 2. Any file in EXPORT_DIR that's NOT in PROMOTED_SET is removed. This is the cleanup of stale artifacts from previous runs. The v1.1 bug
# >>    where fact_capital_adequacy.csv survived is fixed here.
# >>
# >> 3. Staged files are copied to EXPORT_DIR.
# >>
# >> 4. The manifest is recomputed against the EXPORT_DIR contents (not the STAGING_DIR contents) and written last. This ensures the manifest
# >>    reflects the final state.
# >>
# >> 5. The staging directory is removed.
# >>
# >> 6. An assertion verifies that EXPORT_DIR contains exactly SPEC_EXPORTS plus manifest.csv. If any file is missing or extra, the assertion
# >>    fails and the notebook does not complete successfully.
# >>
# >> The mechanism is deliberately transactional: either all expected files are present (and the assertion passes) or the notebook fails loudly. No
# >> partial state survives.
PROMOTED_SET = SPEC_EXPORTS | {'manifest.csv'}

# >> NOTE: The cleanup loop. Any file in EXPORT_DIR that's not in PROMOTED_SET is stale — either from a previous run with different
# >> exports, or from some other process writing to this directory. The WARNING log records what was removed, so a reviewer can see if something
# >> unexpected was cleaned up.
for existing in os.listdir(EXPORT_DIR):
    if existing == os.path.basename(STAGING_DIR):
        continue
    if existing not in PROMOTED_SET:
        os.remove(os.path.join(EXPORT_DIR, existing))
        audit_logger.warning(
            f"Removed stale artifact from previous run: {existing}"
        )

# >> NOTE: The copy forward. Every staged file (except the temporary manifest) is copied to EXPORT_DIR. The shutil.copy2 preserves file
# >> metadata (timestamps, permissions), though the timestamps aren't meaningful after copy.
for fname in os.listdir(STAGING_DIR):
    if fname == 'manifest.csv':
        continue
    src = os.path.join(STAGING_DIR, fname)
    dst = os.path.join(EXPORT_DIR, fname)
    shutil.copy2(src, dst)

# >> NOTE: The final manifest. Recompute hashes against EXPORT_DIR, not STAGING_DIR. The hashes should be identical to the temporary manifest's
# >> hashes (since the files were just copied without modification), but computing against the final location is the correct pattern — the
# >> manifest is a record of what's actually in EXPORT_DIR.
final_manifest_rows = []
for fname in sorted(os.listdir(EXPORT_DIR)):
    if fname == 'manifest.csv':
        continue
    fpath = os.path.join(EXPORT_DIR, fname)
    if not os.path.isfile(fpath):
        continue
    final_manifest_rows.append({
        'file': fname,
        'hash_sha256_16': compute_file_hash(fpath),
        'size_bytes': os.path.getsize(fpath),
        'run_id': RUN_ID,
        'notebook_version': NOTEBOOK_VERSION,
        'timestamp': datetime.now().isoformat(),
    })
pd.DataFrame(final_manifest_rows).to_csv(
    os.path.join(EXPORT_DIR, 'manifest.csv'), index=False)

# >> NOTE: Cleanup. The staging directory is removed after promotion. If the notebook crashes between promotion and cleanup, the staging directory
# >> would be left behind — but the cleanup at the START of the next run would remove it. So a crashed run doesn't leave the staging directory
# >> around indefinitely.
shutil.rmtree(STAGING_DIR)
audit_logger.info("Staging directory cleaned")

# >> NOTE: The promotion assertion. This is the final safety check: after all the copy forward and manifest writing, is EXPORT_DIR in the expected state?
# >>
# >> The assertion checks:
# >> - Missing: files in SPEC_EXPORTS that aren't in EXPORT_DIR. If a copy failed silently, this fires.
# >> - Extra: files in EXPORT_DIR that aren't in SPEC_EXPORTS. If the cleanup loop missed something (or a file appeared between glob and check), this
# >>   fires.
# >>
# >> If either assertion fails, the notebook raises and does not complete. This means an incomplete or contaminated export directory is detected
# >> before any downstream consumer (Power BI) sees it.
actual = set(os.listdir(EXPORT_DIR)) - {'manifest.csv'}
expected = SPEC_EXPORTS
missing = expected - actual
extra = actual - expected
assert not missing, f"Missing exports: {missing}"
assert not extra, f"Unexpected artifacts in {EXPORT_DIR}: {extra}"
audit_logger.info(f"Promotion verified: {len(expected)} exports + manifest")

2026-10-22 13:58:53 | INFO | Staging directory cleaned
2026-10-22 13:58:53 | INFO | Promotion verified: 9 exports + manifest


### SECTION 6: C5 SCOPE GUARD

In [9]:
# NOTE: [E-N06-06] The C5 scope guard. NB06 is a data export — it produces files for a dashboard, not an analytical result. AUC/ROC don't apply.
# 
# The guard checks the notebook's namespace for any `roc_curve` or `auc`variable. If a developer adds code that computes one (e.g., by importing
# sklearn.metrics and computing an AUC for a chart), this assertion fires.
# 
# The guard is deliberately simple — it's a dir() check, not a static analysis. A clever developer could compute AUC without a variable named
# `auc`. But the guard catches the accidental case, which is the common one.
# 
# Combined with E-N06-06's import removal (at the top of the file), the guard makes it essentially impossible to accidentally export an AUC
# value without noticing.
assert 'roc_curve' not in dir() and 'auc' not in dir(), (
    "C5 violation: ROC/AUC must not be produced in NB06."
)

### COMPLETE

In [10]:
print(f"\n{'=' * 60}")
print(f"POWER BI EXPORT v{NOTEBOOK_VERSION} COMPLETE")
print(f"{'=' * 60}")
print(f"\nExported files in {EXPORT_DIR}/:")
for f in sorted(os.listdir(EXPORT_DIR)):
    print(f"  - {f}")

print(f"\n✅ Promotion verified: {len(SPEC_EXPORTS)} exports + manifest")
print(f"✅ C5 scope guard: no ROC/AUC in namespace")
audit_logger.info(f"POWER BI EXPORT v{NOTEBOOK_VERSION} COMPLETE")


POWER BI EXPORT v1.2 COMPLETE

Exported files in powerbi_exports/:
  - data_dictionary.md
  - dim_segment.csv
  - fact_capital_coverage.csv
  - fact_fairness_testing.csv
  - fact_ifrs9_stages.csv
  - fact_loan_portfolio.csv
  - fact_scenario_analysis.csv
  - fact_segment_analysis.csv
  - manifest.csv
  - summary_metrics.csv

✅ Promotion verified: 9 exports + manifest
✅ C5 scope guard: no ROC/AUC in namespace
2026-10-22 13:58:53 | INFO | POWER BI EXPORT v1.2 COMPLETE


## What this notebook establishes, and what it hands off

This notebook produces nine CSV files in `powerbi_exports/`:

- `fact_loan_portfolio.csv` — 100,000 rows, loan-level data
- `fact_segment_analysis.csv` — 15 rows, segment aggregations
- `fact_scenario_analysis.csv` — 4 rows, stress scenario results
- `fact_capital_adequacy.csv` — 5 rows, capital impact by scenario
- `fact_ifrs9_stages.csv` — 3 rows, IFRS 9 stage distribution
- `fact_fairness_testing.csv` — 3 rows, fairness testing results
- `dim_date.csv` — 2,192 rows, date dimension
- `dim_segment.csv` — 5 rows, segment dimension
- `summary_metrics.csv` — 17 rows, key performance indicators

These files are ready to import into Power BI Desktop.

**What the notebook does correctly:**

- The **loan-level fact table** is derived from the data and will reflect
  whatever the portfolio actually contains.
- The **segment analysis table** is computed via `groupby` and reflects the
  data rather than a hardcoded value.
- The **date and segment dimensions** are structurally correct — the right
  shape for a star schema.

**What needs correction before the export is consistent with the analysis:**

Three of the fact tables (fairness, IFRS 9 stages, capital adequacy) currently
report values that do not match the corrected analysis in earlier notebooks.
The scenario table happens to match but is hardcoded rather than derived.

A corrected version would:

1. Derive scenario ECLs from base PD, LGD, and multipliers.
2. Compute capital adequacy on a portfolio-aggregate basis.
3. Recompute the IFRS 9 stage distribution using the same proxy SICR logic
   as Notebook 03.
4. Replace the fairness export with an honest status: either actual
   protected-class results (if available), or a "not assessed" indicator.

**What this notebook demonstrates about the series as a whole:**

The reporting and export layer is where the whole pipeline's discipline is
tested. Every inconsistency introduced upstream — a hardcoded value, a
miscalculated ratio, an invalid test — surfaces here as a dashboard that
disagrees with the written reports.

The notebook's current state reflects the fact that the previous notebooks
were corrected during this review process. Those corrections have not yet
propagated to the export. Bringing them forward is the final step in making
the whole project internally consistent.

**The most important thing this notebook establishes:**

A well-structured export layer is not just a convenience. It is a governance
artifact in its own right. A Power BI dashboard that a CRO relies on must
reflect the same results the compliance report describes. When they differ,
the dashboard becomes a liability — it gives decision-makers a picture that
the analysis does not support.